# The ASN and spin-chain Hamiltonians

This notebook explains Fig. 1, Fig. 2, and Appendices A–B of manuscript v24. Install the checkout with `pip install -e '.[reproduce]'` first, then run all cells. No training or external files are required.

`KharKAN` is the historical Python name of the activation-selection network (ASN). For each input to a layer it mixes analytical activations, then applies a linear map. With `include_abs=False`, the actual implementation orders its library as identity, square, zero. Sparsemax can give exact zero mixture weights.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from experiments.paper import ROOT
print('Repository:', ROOT)

In [ ]:
import torch
from nmrkan.models import KharKAN
model = KharKAN((2, 2, 1), include_abs=False).double()
model.switch_to_sparsemax()
with torch.no_grad():
    first, second = model.layers
    first.act_logits[:] = torch.tensor([[50., -50., -50.]] * 2)
    second.act_logits[:] = torch.tensor([[-50., 50., -50.]] * 2)
    first.linear.weight[:] = torch.tensor([[1., 1.], [1., -1.]])
    second.linear.weight[:] = torch.tensor([[.25, -.25]])
    first.linear.bias.zero_(); second.linear.bias.zero_()
x = torch.tensor(np.random.default_rng(42).normal(size=(100, 2)), dtype=torch.float64)
with torch.no_grad():
    product = model(x).numpy().ravel()
np.testing.assert_allclose(product, x[:, 0].numpy()*x[:, 1].numpy(), atol=1e-14)
print(model.symbolic_formula())

The two hidden values are $x_1+x_2$ and $x_1-x_2$. Squaring and subtracting with weights $1/4$ and $-1/4$ gives $x_1x_2$ exactly. This is a mathematical construction, not a fitted neural result.

## Hamiltonians and units
The paper uses $r_i=\Delta J_i/J$ and descending $E/J$. Setting $J=1$ constructs the dimensionless matrix. For the physical negative $J$, physical energy order reverses. The molecular reference uses a different basis permutation with the same spectrum.

In [ ]:
from data_generation.nmr_datagen import ham_reduced_symmetric
from data_generation.eigenvalues_4CH2 import Ham4CH2s3Sand1S
from nmrkan.physics.three_ch2 import self_check
from experiments.paper import exact_three, exact_four
print('Three-site block:')
print(ham_reduced_symmetric(1, 1, 1, .2, .3, 0))
print('Four-site block:')
print(Ham4CH2s3Sand1S(1, .2))
print(self_check())
np.testing.assert_allclose(exact_three([[.2,.3]]).sum(), -3)
y = exact_four(np.linspace(0, 3, 101)[:, None])
np.testing.assert_allclose(y + y[:, ::-1], -2, atol=2e-14)

Next: [three-site results](01_three_site_paper.ipynb), [four-site results](02_four_site_paper.ipynb), and [real molecules](03_real_molecules.ipynb). The original diagram artwork remains in `paper/manuscript_v24.pdf`.